# Homework: Round-Object Recognition

**Course:** Medical Image Analysis & Processing (MIAP)

**Topic:** Object description &mdash; shape descriptors

**Input image:** `pillsetc.png` (provided next to this notebook)

---

## Learning objectives

After completing this assignment you should be able to:

1. Segment objects from a noisy background using **Otsu thresholding** and basic morphological cleanup.
2. Extract object **contours** and compute simple **shape descriptors** (area, perimeter, centroid).
3. Implement and interpret the **roundness / circularity** descriptor

   $$\alpha \;=\; \dfrac{4\pi A}{P^{2}}\,, \qquad 0 \le \alpha \le 1$$

   where $A$ is the object area and $P$ is its perimeter. A perfect digital disk has $\alpha = 1$; the more elongated or irregular the boundary, the smaller $\alpha$.
4. Threshold $\alpha$ to **classify** which objects are circular and visualise the decision.

## Deliverables

- This notebook with all `TODO` cells completed and run end-to-end.
- A short written answer to each **Question** in the discussion section.
- The final figure showing the original image with circular objects highlighted.

> Look out for the orange `# --- TODO N.k ---` markers. There are **9 TODOs** in total.


## 1. Setup

Run the cell below to import everything we need. If `opencv-python`, `scikit-image` or `scipy` are not installed, run

```
pip install opencv-python scikit-image scipy matplotlib numpy
```

once in your environment.

### TODO 1.1
Add the missing imports. You will need: `cv2`, `numpy as np`, `matplotlib.pyplot as plt`, `scipy.ndimage as ndi`, and `morphology` from `skimage`.

In [ ]:
# --- TODO 1.1 -------------------------------------------------------------
# Import the libraries listed in the markdown cell above.
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import morphology
# --------------------------------------------------------------------------

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["image.cmap"]    = "gray"


## 2. Load the image

The image `pillsetc.png` lives next to this notebook. OpenCV reads images in **BGR** order, but `matplotlib` expects **RGB**. We also need a grayscale copy for thresholding.

### TODO 2.1
Load the image with `cv2.imread`, then build:
- `rgb`  &mdash; the RGB version, for display
- `gray` &mdash; an 8-bit grayscale version, for thresholding

Use `cv2.cvtColor` with the appropriate conversion flags
(`cv2.COLOR_BGR2RGB`, `cv2.COLOR_BGR2GRAY`).

### TODO 2.2
Display `rgb` and `gray` side-by-side using `plt.subplots(1, 2, ...)`. Give each panel a title and turn the axes off.

In [ ]:
IMG_PATH = "pillsetc.png"

# --- TODO 2.1 -------------------------------------------------------------
bgr  = cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)  # cv2.imread(IMG_PATH, cv2.IMREAD_COLOR)
assert bgr is not None, f"Could not read {IMG_PATH}.  Is it next to this notebook?"
rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)   # convert BGR -> RGB
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)  # convert BGR -> grayscale
# --------------------------------------------------------------------------

print("Image size :", rgb.shape)
print("Gray dtype :", gray.dtype)

# --- TODO 2.2 -------------------------------------------------------------
# Display the original RGB image and the grayscale version side by side.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(rgb)                                   # show rgb
ax[0].set_title("Original RGB")
ax[1].imshow(gray, cmap='gray')                     # show gray, cmap='gray'
ax[1].set_title("Grayscale")
for a in ax: a.axis("off")
plt.show()
# --------------------------------------------------------------------------


## 3. Otsu thresholding

Otsu's method picks the threshold that **maximises the between-class variance** of the histogram, which is a good automatic choice when the histogram is roughly bimodal (background vs. objects).

> **Question 1.** Inspect the grayscale image above. Is the foreground (objects) brighter or darker than the background? Why does that matter for Otsu?

In the grayscale image, the foreground objects appear darker than the background, which is relatively bright or light-colored. This matters for Otsu’s method because the algorithm automatically determines a threshold by maximizing the variance between two classes , typically the darker and brighter regions. Otsu assumes a bimodal histogram where one peak represents the background and the other represents the foreground. However, Otsu itself does not inherently know which class should be labeled as “foreground” or “background”; it simply separates the two intensity groups. Depending on the intensity distribution, the resulting binary image might assign white (255) to the background and black (0) to the objects, or vice versa. Since we want the objects to be white for easier subsequent processing, we need to check the output and potentially invert it using cv2.bitwise_not if the background ended up as white. This ensures consistency in our pipeline where foreground pixels are always represented as white.

### TODO 3.1
Use `cv2.threshold(..., cv2.THRESH_BINARY + cv2.THRESH_OTSU)` to obtain the threshold value and the binary image. Then make sure that **foreground pixels are 255 (white) and background is 0 (black)** &mdash; if Otsu happened to invert it, flip the mask with `cv2.bitwise_not`.

In [ ]:
# --- TODO 3.1 -------------------------------------------------------------
# 1) Apply Otsu thresholding to `gray`. The function returns (threshold, binary).
otsu_thr, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 2) If the background is white (mean > 127), invert so that objects = white.
if binary.mean() > 127:
    binary = cv2.bitwise_not(binary)
# --------------------------------------------------------------------------
print(f"Otsu threshold value = {otsu_thr}")
plt.imshow(binary, cmap="gray")
plt.title(f"Binary image after Otsu (thr={otsu_thr:.0f})")
plt.axis("off")
plt.show()


## 4. Morphological cleanup

Real images are noisy: there will be a few stray foreground pixels in the background, small holes inside objects, and frayed boundaries. We clean these up with three classic morphological steps:

| step | what it does | function |
|------|--------------|----------|
| remove small specks | drop connected components below a few hundred pixels | `skimage.morphology.remove_small_objects` |
| binary closing      | close small gaps along boundaries (dilate $\to$ erode)    | `skimage.morphology.closing` with a disk SE |
| fill holes          | turn enclosed background regions into foreground         | `scipy.ndimage.binary_fill_holes` |

> **Question 2.** What would happen to the **roundness** of an object if we *skipped* the hole-filling step (a ring-shaped object stays a ring)? Sketch the answer in 1&ndash;2 sentences after running the next cell.

**A filled circle is maximally round, but a ring has a much longer perimeter for the same outer area, making it appear less circular by this measure.**

### TODO 4.1
Convert `binary` (a `0/255` `uint8` array) into a boolean mask.

### TODO 4.2
Apply the three cleanup steps to `mask_bool` &mdash; `remove_small_objects`, `closing` (with a `disk(5)` structuring element), and `binary_fill_holes`.

### TODO 4.3
Display the **before** mask (`binary`) and the **after** mask (`clean`) side by side so you can verify the cleanup actually did something.

In [ ]:
# --- TODO 4.1 -------------------------------------------------------------
mask_bool = binary > 0                       # convert {0,255} -> bool
# --------------------------------------------------------------------------

# --- TODO 4.2 -------------------------------------------------------------
# (a) remove small objects (use min_size=200)
mask_bool = morphology.remove_small_objects(mask_bool, min_size=200)
# (b) binary closing with a disk-shaped structuring element of radius 5
mask_bool = morphology.closing(mask_bool, morphology.disk(5))
# (c) fill internal holes
mask_bool = ndi.binary_fill_holes(mask_bool)
# --------------------------------------------------------------------------

clean = (mask_bool.astype(np.uint8)) * 255

# --- TODO 4.3 -------------------------------------------------------------
# Show `binary` and `clean` side by side; give each panel a descriptive title.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(binary, cmap="gray")
ax[0].set_title("Before cleanup (Otsu binary)")
ax[1].imshow(clean, cmap="gray")
ax[1].set_title("After morphological cleanup")
for a in ax: a.axis("off")
plt.show()
# --------------------------------------------------------------------------


## 5. Find object contours

We look for **external** contours only (`cv2.RETR_EXTERNAL`) so that holes inside an object don't count as separate objects. We use `cv2.CHAIN_APPROX_NONE` so every boundary pixel is kept &mdash; this matters because we will compute the perimeter from this contour.

### TODO 5.1
Call `cv2.findContours` on the cleaned mask. Choose the retrieval mode and the chain-approximation method according to the discussion above.

In [ ]:
# --- TODO 5.1 -------------------------------------------------------------
# Hint: cv2.findContours returns (contours, hierarchy) on OpenCV 4+.
contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
# --------------------------------------------------------------------------

print(f"Number of contours found: {len(contours)}")

## 6. Compute the roundness descriptor

For every contour we compute

$$A \;=\; \text{contour area}\;\;(\text{px}^2),\qquad
P \;=\; \text{contour perimeter}\;\;(\text{px}),\qquad
\boxed{\;\alpha \;=\; \dfrac{4\pi A}{P^{2}}\;}$$

and the centroid $(c_x, c_y)$ from the image moments

$$c_x = \dfrac{m_{10}}{m_{00}},\qquad c_y = \dfrac{m_{01}}{m_{00}}.$$

OpenCV gives us all the building blocks: `cv2.contourArea`, `cv2.arcLength`, and `cv2.moments`.

### TODO 6.1
Fill in the lines that compute `area`, `perim`, and `roundness`.

### TODO 6.2
Compute the centroid `(cx, cy)` from the spatial moments `M["m00"]`, `M["m10"]`, `M["m01"]` returned by `cv2.moments`. Guard against `M["m00"] == 0` (a degenerate contour).

In [ ]:
ROUNDNESS_THRESHOLD = 0.9   # try other values later (Question 4)
results = []

for c in contours:
    # --- TODO 6.1 ---------------------------------------------------------
    area      = cv2.contourArea(c)                 # hint: cv2.contourArea
    perim     = cv2.arcLength(c, closed=True)      # hint: cv2.arcLength(c, closed=True)
    if area < 200 or perim == 0:
        continue
    roundness = (4 * np.pi * area) / (perim ** 2)  # hint: 4 * pi * A / P^2
    # ----------------------------------------------------------------------

    # --- TODO 6.2 ---------------------------------------------------------
    M  = cv2.moments(c)
    cx = M["m10"] / M["m00"] if M["m00"] != 0 else 0  # hint: m10 / m00 (guard against 0)
    cy = M["m01"] / M["m00"] if M["m00"] != 0 else 0  # hint: m01 / m00 (guard against 0)
    # ----------------------------------------------------------------------

    results.append(dict(area=area, perim=perim,
                        roundness=roundness,
                        cx=cx, cy=cy, contour=c))

# Sort largest-roundness first for the printout
results.sort(key=lambda r: -r["roundness"])

print(f"{'idx':>3} {'area':>10} {'perim':>10} {'alpha':>8}  centroid")
for i, r in enumerate(results, 1):
    flag = "  <- round" if r["roundness"] >= ROUNDNESS_THRESHOLD else ""
    print(f"{i:>3} {r['area']:>10.1f} {r['perim']:>10.2f} {r['roundness']:>8.4f}"
          f"  ({r['cx']:6.1f},{r['cy']:6.1f}){flag}")


## 7. Visualise the decision

We draw every detected contour on a copy of the original image. Round objects (those with $\alpha \ge$ `ROUNDNESS_THRESHOLD`) are highlighted in **red**; the rest are drawn in **yellow** for context. The centroid of each round object is also marked with a small filled red dot inside a red ring.

### TODO 7.1
Loop over `results` and draw:

- the contour, in **red** if the object is round and **yellow** otherwise (`cv2.drawContours`),
- a red ring of radius 12 (line thickness 3) around the centroid of every round object (`cv2.circle`),
- a small filled red dot of radius 3 at the centroid of every round object (`cv2.circle` with thickness `-1`).

Remember that `cv2.circle` expects the centre as `(int(cx), int(cy))`.

In [ ]:
overlay = rgb.copy()
RED, YELLOW = (255, 0, 0), (255, 255, 0)

# --- TODO 7.1 -------------------------------------------------------------
for r in results:
    color = RED if r["roundness"] >= ROUNDNESS_THRESHOLD else YELLOW
    cv2.drawContours(overlay, [r["contour"]], -1, color, 3)

    if r["roundness"] >= ROUNDNESS_THRESHOLD:
        center = (int(r["cx"]), int(r["cy"]))
        cv2.circle(overlay, center, 12, RED, 3)      # outer red ring at centroid
        cv2.circle(overlay, center, 3, RED, -1)      # filled red dot at centroid
# --------------------------------------------------------------------------

plt.figure(figsize=(8, 7))
plt.imshow(overlay)
plt.title(f"Round objects (alpha >= {ROUNDNESS_THRESHOLD}) shown in red, "
          f"others in yellow")
plt.axis("off")
plt.show()


## 8. Per-object diameter estimate

If we *assume* an object is a perfect disk, we can recover its diameter from its area:

$$d \;=\; 2\sqrt{A/\pi}$$

This is a useful sanity check &mdash; e.g. for the two pills you should get diameters that look right by eye.

### TODO 8.1
For every object that was classified as round, compute and print

- its index `i`,
- its centroid `(cx, cy)`,
- its area `A`,
- its estimated diameter `d`.

In [ ]:
# --- TODO 8.1 -------------------------------------------------------------
print(f"{'idx':>3} {'centroid':>18} {'area':>10} {'diameter (px)':>15}")
for i, r in enumerate(results, 1):
    if r["roundness"] >= ROUNDNESS_THRESHOLD:
        diameter = 2 * np.sqrt(r["area"] / np.pi)
        print(f"{i:>3} ({r['cx']:>7.1f}, {r['cy']:>6.1f}) {r['area']:>10.1f} {diameter:>15.1f}")
# --------------------------------------------------------------------------


## 9. Discussion questions

Write your answers in the markdown cell at the bottom of this section.

1. **(Otsu)** Which side of the histogram corresponds to the objects in this image, and how did you tell that from the binary mask?
2. **(Morphology)** What goes wrong if you skip the hole-filling step?  Hint: think about ring-shaped objects (the rubber bands in this picture).
3. **(Descriptor sensitivity)** The two rubber bands have $\alpha \approx 0.88$, just below the 0.9 cut-off. Why is this even though they look very circular?  *Hint*: consider how small irregularities of the boundary contribute to $P^{2}$ vs. $A$.
4. **(Threshold choice)** Re-run the notebook with `ROUNDNESS_THRESHOLD = 0.85` and with `0.95`. Which objects switch label, and what does that tell you about choosing a single global threshold for $\alpha$?
5. **(Robustness)** Suggest one shape descriptor *other* than $\alpha = 4\pi A/P^{2}$ that would be **less** sensitive to boundary noise.  (You don't have to implement it &mdash; one or two sentences are enough.)


### Your answers

*Write your answers here. Double-click this cell to edit.*

1. 

The objects correspond to the higher intensity side of the histogram (≥145). Since binary.mean() = 33 , only ~13% of pixels are white (255), meaning the objects are the minority bright regions that became white, while the darker background (~87%) became black.

2. 
Skipping hole-filling leaves the rubber bands as hollow rings instead of filled disks. This dramatically increases their perimeter 𝑃 relative to area 𝐴, causing roundness 
$α=4πA/P^2$ to drop significantly, potentially misclassifying circular rings as non-round objects.

3. 

Small boundary irregularities (noise, pixelation) add linearly to perimeter 𝑃, but roundness depends on $p^2$ in the denominator. Even minor roughness causes $p^2$ to grow quadratically while area 𝐴 remains nearly constant, making 𝛼 drop disproportionately. The rubber bands’ boundaries have slight irregularities that push them just below 0.9.

4. 

At 0.85, the rubber bands (𝛼≈0.88) would be classified as round; at 0.95, they’d be excluded along with some irregular pills. This shows that a single global threshold is sensitive to minor shape variations and requires careful tuning based on whether we prioritize precision (fewer false positives) or recall (catching all circular objects).

5.

Solidity (object area / convex hull area) is more robust to boundary noise. The convex hull smooths over small irregularities and indentations, so minor boundary perturbations barely affect the ratio, unlike perimeter-based measures where every pixel of noise contributes directly to 𝑃.


## 10. Optional bonus (ungraded)

### TODO 10.1 (bonus)
Annotate the final overlay image with the diameter (in pixels) **next to each round object's centroid**, using `cv2.putText`. Choose a font, scale, colour, and thickness that are readable on this image. Re-display the result.

### TODO 10.2 (bonus, harder)
Replace the fixed cut-off `ROUNDNESS_THRESHOLD = 0.9` with an **automatic** rule based on, for example, Otsu's method applied to the list of $\alpha$-values across all detected objects. Discuss in two sentences whether this works better or worse than the fixed value.


In [ ]:
# 10.1

overlay_annotated = rgb.copy()

for obj in results:
    cx, cy = obj['cx'], obj['cy']
    area = obj['area']
    roundness = obj['roundness']
    contour = obj['contour']
    
    if roundness >= ROUNDNESS_THRESHOLD:
        color = (255, 0, 0)  
    else:
        color = (255, 255, 0) 
    
    cv2.drawContours(overlay_annotated, [contour], -1, color, 2)
    
    if roundness >= ROUNDNESS_THRESHOLD:

        cv2.circle(overlay_annotated, (int(cx), int(cy)), 12, (255, 0, 0), 3)
        cv2.circle(overlay_annotated, (int(cx), int(cy)), 3, (255, 0, 0), -1)
        
        diameter = 2 * np.sqrt(area / np.pi)
        
        text_x = int(cx) + 15
        text_y = int(cy) + 5
        
        text = f"{diameter:.1f}"

        cv2.putText(overlay_annotated, text, (text_x, text_y),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
        cv2.putText(overlay_annotated, text, (text_x, text_y),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

plt.figure(figsize=(12, 8))
plt.imshow(overlay_annotated)
plt.title('Round Objects with Diameter Annotations')
plt.axis('off')
plt.tight_layout()
plt.show()
